In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

account = "bikestores@myolympicdatastorage.dfs.core.windows.net"
bronze = f"abfss://{account}/bronze"
silver = f"abfss://{account}/silver"

log_rows = []

def log(table, step, rows_in, rows_rejected, rows_out):
    log_rows.append((table, step, rows_in, rows_rejected, rows_out))
    print(f"[{table}] {step}: read={rows_in}, rejected={rows_rejected}, written={rows_out}")

In [ ]:
t = "customers"
df = spark.read.parquet(f"{bronze}/sales/{t}")
n_read = df.count()

# 1. duplicates hatana
df = df.dropDuplicates(["customer_id"])
log(t, "dedup", n_read, n_read - df.count(), df.count())

# 2. types theek karna + text clean karna
df = (df
      .withColumn("customer_id", F.col("customer_id").cast("int"))
      .withColumn("first_name", F.trim(F.initcap("first_name")))
      .withColumn("last_name", F.trim(F.initcap("last_name")))
      .withColumn("email", F.lower(F.trim("email"))))

# 3. missing values
df = df.fillna({"phone": "Unknown"})

# 4. data quality check: email valid hona chahiye
valid = df.filter(F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))
bad = df.count() - valid.count()
log(t, "email_check", df.count(), bad, valid.count())

valid.write.mode("overwrite").format("delta").save(f"{silver}/{t}")

In [ ]:
t = "orders"
df = spark.read.parquet(f"{bronze}/sales/{t}")
n_read = df.count()

df = df.dropDuplicates(["order_id"])
log(t, "dedup", n_read, n_read - df.count(), df.count())

df = (df
      .withColumn("order_id", F.col("order_id").cast("int"))
      .withColumn("customer_id", F.col("customer_id").cast("int"))
      .withColumn("order_date", F.to_date("order_date"))
      .withColumn("shipped_date", F.to_date("shipped_date")))

# quality check: order_date valid, aur shipped_date order_date se pehle na ho
valid = df.filter(
    F.col("order_date").isNotNull() &
    (F.col("order_date") <= F.current_date()) &
    (F.col("shipped_date").isNull() | (F.col("shipped_date") >= F.col("order_date")))
)
log(t, "date_check", df.count(), df.count() - valid.count(), valid.count())

valid.write.mode("overwrite").format("delta").save(f"{silver}/{t}")

In [ ]:
t = "order_items"
df = spark.read.parquet(f"{bronze}/sales/{t}")
n_read = df.count()

df = df.dropDuplicates(["order_id", "item_id"])
log(t, "dedup", n_read, n_read - df.count(), df.count())

df = (df
      .withColumn("quantity", F.col("quantity").cast("int"))
      .withColumn("list_price", F.col("list_price").cast("decimal(10,2)"))
      .withColumn("discount", F.col("discount").cast("decimal(4,2)")))

df = df.fillna({"discount": 0})

# quality checks: quantity > 0, price > 0, discount 0 se 1 ke beech
valid = df.filter((F.col("quantity") > 0) & (F.col("list_price") > 0) &
                  (F.col("discount").between(0, 1)))
log(t, "qty_price_check", df.count(), df.count() - valid.count(), valid.count())

valid.write.mode("overwrite").format("delta").save(f"{silver}/{t}")

In [ ]:
t = "products"
df = spark.read.parquet(f"{bronze}/production/{t}")
n_read = df.count()

df = df.dropDuplicates(["product_id"])
log(t, "dedup", n_read, n_read - df.count(), df.count())

df = (df
      .withColumn("product_id", F.col("product_id").cast("int"))
      .withColumn("list_price", F.col("list_price").cast("decimal(10,2)")))

valid = df.filter(F.col("list_price") > 0)
log(t, "price_check", df.count(), df.count() - valid.count(), valid.count())

valid.write.mode("overwrite").format("delta").save(f"{silver}/{t}")

In [ ]:
log_df = spark.createDataFrame(log_rows, ["table", "step", "rows_read", "rows_rejected", "rows_written"])
display(log_df)

In [ ]:
display(dbutils.fs.ls(silver))